# Pipeline Overview

**Enhancing Food Label Transparency for Filipino Consumers Through AI-Based Ingredient Interpretation**

This document maps the entire notebook pipeline, its two deliverables, the ground-truth
hierarchy, and the methodology guarantees that all downstream notebooks follow.

## Two Deliverables

1. **Semantic ingredient classification** — a MobileBERT model that maps ingredient
   phrases to 40 semantic categories (used for *interpreting* complex ingredient lists).
   Trained and evaluated in Phase B.
2. **FDA Big-8 allergen detection** — a hybrid **rule-based + MobileBERT** system that
   flags the 8 major allergens (milk, eggs, peanuts, tree_nuts, soy, wheat, fish,
   shellfish) from ingredient text. Built across Phases C–G.

## Pipeline Map

Run the notebooks in order. Each phase consumes the artifacts of the previous one.

| # | Notebook | Phase | Purpose |
|---|----------|-------|---------|
| 01 | `01_extraction` | Data | Pull product data (OFF) |
| 02 | `02_cleaning` | Data | Clean ingredient text |
| 03 | `03_ingredient_parsing` | Data | Tokenize / parse ingredients |
| 04 | `04_semantic_labeling` | Semantic (D1) | Rule-based semantic labels |
| 05 | `05_semantic_model` | Semantic (D1) | Train semantic MobileBERT |
| 06 | `06_allergen_labeling` | Ground truth (D2) | Rule-based allergen labels + **official-tag validation** |
| 07 | `07_training_data_preparation` | Data prep (D2) | **Stratified split, then augment train only** |
| 08 | `08_allergen_training` | Training (D2) | BCE / Focal / ASL / ensemble; **validation-based selection** |
| 09 | `09_allergen_evaluation` | Evaluation (D2) | Thresholds on val, **held-out test once**, ML-vs-official |
| 10 | `10_hybrid_evaluation` | Hybrid (D2) | Rule vs ML vs hybrid; honest delta analysis |
| 11 | `11_model_export` | Deployment | ONNX export + validation + latency |
| 12 | `12_mobile_benchmark` | Deployment | ONNX Runtime Mobile benchmark |

## Ground-Truth Hierarchy

For the allergen task, three sources of truth exist and are used at different points:

1. **Official OFF tags** (`allergens_tags`) — declared by the manufacturer on the label.
   This is the strongest external signal. Used to *validate* our rule engine (nb 06)
   and our ML model (nb 09), never to train the ML model.
2. **Rule-based detection** (`detected_allergens`) — our keyword/negation/exemption
   engine. After Phase-4 refinements it reaches **55.21 % exact agreement** on the 96
   products with explicit statements (target > 50 %).
3. **Consensus labels** — intersection of rule-based and official tags, used as the
   training target for the ML model. This is a deliberate choice (labels are *generated*
   by the rule engine); the official-tag evaluations in 06/09 quantify how well that
   ground truth agrees with manufacturer declarations.

## Methodology Guarantees

The restructure bakes in five methodological guarantees:

1. **No augmentation leakage.** The stratified split happens *first* (nb 07); all
   augmentation (generic synonym, targeted oversampling, counter-examples) is applied
   **only to the training split**. Validation and test sets contain only original products.
2. **No test-set model selection.** Model architecture/loss selection (nb 08) and
   threshold calibration (nb 09) use the **validation** set only. The held-out test set
   is evaluated **once**, at the end of nb 09, and those numbers are the final reported
   ones.
3. **Honest hybrid reporting.** nb 10 reports *how often* the rule-based overrides
   actually change an ML prediction, and breaks the changes down into helps vs hurts —
   rather than presenting a config that changes nothing as an improvement.
4. **Ground-truth validation in the labeling phase.** Rule-vs-official agreement is
   measured where the labels are built (nb 06), so label quality is known *before* training.
5. **Single source of truth for thresholds.** `models/hybrid_config.json` is written by
   nb 09 (thresholds tuned on val) and consumed by nb 10 and deployment.

## Running Order

```
jupyter lab
# from the notebooks/ directory (relative imports rely on cwd)
01 → 02 → 03 → 04 → 05 → 06 → 07 → 08 → 09 → 10 → 11 → 12
```

Training (nb 08) requires a GPU; the rest runs on CPU. Notebooks 07/08/09/10 were
restructured without re-running — execute them to regenerate the numbers.